In [1]:
# Importing Modules
import pandas as pd
import numpy as np
from src.utils.smiles2morganfp import MorganFP
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import clone

In [2]:
######################## DATA-1 ##################################
# Loading ESOL data
esol_train_data = pd.read_csv("data/train/ESOL.csv")
esol_test_data = pd.read_csv("data/test/ESOL.csv")

# Generate ESOL FP
esol_train_fp = MorganFP(esol_train_data["smiles"], bits=1024)
esol_train_fp["smiles"] = esol_train_fp.index
esol_train_fp = esol_train_fp.merge(esol_train_data, on="smiles")
esol_test_fp = MorganFP(esol_test_data["smiles"], bits=1024)
esol_test_fp["smiles"] = esol_test_fp.index
esol_test_fp = esol_test_fp.merge(esol_test_data, on="smiles")

######################## DATA-2 ##################################
# Loading RT data
rt_train_data = pd.read_csv("data/train/RT.csv")
rt_test_data = pd.read_csv("data/test/RT.csv")

# Generate RT FP
rt_train_fp = MorganFP(rt_train_data["smiles"], bits=1024)
rt_train_fp["smiles"] = rt_train_fp.index
rt_train_fp = rt_train_fp.merge(rt_train_data, on="smiles")
rt_test_fp = MorganFP(rt_test_data["smiles"], bits=1024)
rt_test_fp["smiles"] = rt_test_fp.index
rt_test_fp = rt_test_fp.merge(rt_test_data, on="smiles")

######################## DATA-3 ##################################
# Loading Lipophilicity data
lipophilicity_train_data = pd.read_csv("data/train/Lipophilicity.csv")
lipophilicity_test_data = pd.read_csv("data/test/Lipophilicity.csv")

# Generate lipophilicity FP
lipophilicity_train_fp = MorganFP(lipophilicity_train_data["smiles"], bits=1024)
lipophilicity_train_fp["smiles"] = lipophilicity_train_fp.index
lipophilicity_train_fp = lipophilicity_train_fp.merge(lipophilicity_train_data, on="smiles")
lipophilicity_test_fp = MorganFP(lipophilicity_test_data["smiles"], bits=1024)
lipophilicity_test_fp["smiles"] = lipophilicity_test_fp.index
lipophilicity_test_fp = lipophilicity_test_fp.merge(lipophilicity_test_data, on="smiles")

######################## DATA-4 ##################################
# Loading B3DB data
b3db_train_data = pd.read_csv("data/train/B3DB.csv")
b3db_test_data = pd.read_csv("data/test/B3DB.csv")

# Generate B3DB FP
b3db_train_fp = MorganFP(b3db_train_data["smiles"], bits=1024)
b3db_train_fp["smiles"] = b3db_train_fp.index
b3db_train_fp = b3db_train_fp.merge(b3db_train_data, on="smiles")
b3db_test_fp = MorganFP(b3db_test_data["smiles"], bits=1024)
b3db_test_fp["smiles"] = b3db_test_fp.index
b3db_test_fp = b3db_test_fp.merge(b3db_test_data, on="smiles")

######################## DATA-5 ##################################
# Loading freesolv data
freesolv_train_data = pd.read_csv("data/train/FreeSolv.csv")
freesolv_test_data = pd.read_csv("data/test/FreeSolv.csv")

# Generate freesolv FP
freesolv_train_fp = MorganFP(freesolv_train_data["smiles"], bits=1024)
freesolv_train_fp["smiles"] = freesolv_train_fp.index
freesolv_train_fp = freesolv_train_fp.merge(freesolv_train_data, on="smiles")
freesolv_test_fp = MorganFP(freesolv_test_data["smiles"], bits=1024)
freesolv_test_fp["smiles"] = freesolv_test_fp.index
freesolv_test_fp = freesolv_test_fp.merge(freesolv_test_data, on="smiles")

In [3]:
# Function to run ML training and testing
def RunML(model_template, train_data, test_data, dataName, modelName):

	rmse_list = []
	num_runs = 3

	for i in range(num_runs):

		# Data
		X_train = train_data.drop(["smiles", "target"], axis=1).to_numpy()
		y_train = train_data["target"].to_numpy()
		X_test = test_data.drop(["smiles", "target"], axis=1).to_numpy()
		y_test = test_data["target"].to_numpy()

		model = clone(model_template)
	   
		if hasattr(model, 'random_state'):
			model.random_state = i + 1

		# Model training
		model.fit(X_train, y_train)

		# Model testing
		y_pred = model.predict(X_test)

		rmse_run = root_mean_squared_error(y_test, y_pred)
		
		rmse_list.append(rmse_run)

	rmse_mean, rmse_std = np.mean(rmse_list), np.std(rmse_list)
	rmse_str = f"{rmse_mean:.3f} ({rmse_std:.3f})"

	# Return results
	return pd.DataFrame({ 
		"Data": [dataName], 
		"Model": [modelName],
		"RMSE": [rmse_str]
	})

In [4]:
# Data dict
datasets = {"ESOL":{"train":esol_train_fp, "test":esol_test_fp},
            "Lipophilicity":{"train":lipophilicity_train_fp, "test":lipophilicity_test_fp},
            "RT":{"train":rt_train_fp, "test":rt_test_fp},
            "B3DB":{"train":b3db_train_fp, "test":b3db_test_fp},
            "FreeSolv":{"train":freesolv_train_fp, "test":freesolv_test_fp}}

# Model dict
model_dict = {
    "LR": LinearRegression(), 
    "SVM": SVR(),
    "RF": RandomForestRegressor(random_state=42),
    "XGB": XGBRegressor(random_state=42, objective='reg:squarederror', n_jobs=16)
}

In [5]:
# List to store results
temp_out = []

# Loop for models
for modelName, model in model_dict.items():
    # Loop for dataset
    for dataName, data in datasets.items():
        # Run Analysis for model and dataset
        params_df = pd.read_csv(f"results/Output_Hyperparameter_Optimization_ML_{dataName}.csv")
        params = eval(params_df[params_df["Model"]== modelName]["Model Params"].to_list()[0])
        model =  model.set_params(**params)
        temp_out.append(RunML(model, data["train"], data["test"], dataName, modelName))

# Final output
ML_out = pd.concat(temp_out, ignore_index=True)
ML_out.to_csv("results/Output_ML_Analysis.csv")
ML_out

,Data,Model,RMSE
0,ESOL,LR,4.380 (0.000)
1,Lipophilicity,LR,1.011 (0.000)
2,RT,LR,601.600 (0.000)
3,B3DB,LR,1.426 (0.000)
4,FreeSolv,LR,2.116 (0.000)
5,ESOL,SVM,1.128 (0.000)
6,Lipophilicity,SVM,0.730 (0.000)
7,RT,SVM,120.555 (0.000)
8,B3DB,SVM,0.471 (0.000)
9,FreeSolv,SVM,1.512 (0.000)
